#### 앙상블(Ensemble)
- 여러개의 분류모델을 조합해서 더 나은 성능을 내는 방법
- Decision Tree 모델을 증가시켜 나온 랜덤포레스트가 대표적임

#### 랜덤포레스트(Random Forest)
##### 랜덤포레스트 훈련방법
- 부트스트랩 샘플 : 중복을 허용하는 샘플링 방법
- 샘플링 후에 샘플을 복구하고 다시 샘플링 하는 방법입니다.
- 이와 같이 진행하는 이유는 결정트리에서 과대적합을 방지할 수 있기 때문이다.
- 각 결정트리에서 나오는 확률의 합을 트리갯수로 나누어서 결정짓는 모델
- 부트스트랩의 샘플링 갯수 : 특성의 갯수의 제곱근

In [1]:
import pandas as pd

In [2]:
wine = pd.read_csv("../Data/wine.csv")

In [3]:
# Feature와 Target
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [5]:
# Train과 Test
from sklearn.model_selection import train_test_split

In [6]:
train_input, test_input, train_target, test_target = \
    train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=42,
        stratify=target
    )

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

In [8]:
rf = RandomForestClassifier(
    n_jobs= -1 # 내 PC의 모든 가용한 자원 사용
)
scores = cross_validate(
        rf,
        train_input,
        train_target,
        return_train_score=True,
        n_jobs= -1
)

In [12]:
scores

{'fit_time': array([0.21839499, 0.3740983 , 0.24064708, 0.37294888, 0.21948099]),
 'score_time': array([0.04252148, 0.02647161, 0.05642653, 0.02588677, 0.04278398]),
 'test_score': array([0.90192308, 0.89807692, 0.88354187, 0.88546679, 0.87680462]),
 'train_score': array([0.99807554, 0.99807554, 0.998076  , 0.998076  , 0.9983165 ])}

In [13]:
print('Train :', scores['train_score'].mean())
print('Valid :', scores['test_score'].mean())

Train : 0.9981239129904033
Valid : 0.889162656400385


In [14]:
rf.fit(train_input, train_target)
rf.score(test_input, test_target)

0.8915384615384615

In [15]:
# 중요 Feature
print(rf.feature_importances_)

[0.23653512 0.49728609 0.26617878]


----
#### Extra Tree
- 기본적으로 100개의 트리를 사용
- 노드 분할시 특성의 제곱근의 갯수를 사용
- 특성의 선택을 랜덤하게 선택한다.
- 특성의 선택을 랜덤하게 하므로 속도는 랜덤포레스트보다 빠르다.

In [17]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs= -1)
scores = cross_validate(
        et,
        train_input,
        train_target,
        return_train_score=True,
        n_jobs= -1
)

print('Train :', scores['train_score'].mean())
print('Valid :', scores['test_score'].mean())

Train : 0.9981720130385032
Valid : 0.8897399496557341


----
#### Gradient Boosting
- 가장 유명한 알고리즘중 하나이다.
- 경사하강법 처럼 손실함수를 사용.
- 손실함수를 보고 트리를 추가하여 최적의 값 도출하는 방법이다.
- max depth는 3으로 제어됨 -> 과대적합 방지
- 단점은 손실함수를 확인하고 트리는 추가하는 모델이므로 n_jobs(병렬처리)를 할수 없다.

In [21]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier()

scores = cross_validate(
        gb,
        train_input,
        train_target,
        return_train_score=True,
        n_jobs= -1
)

print('Train :', scores['train_score'].mean())
print('Valid :', scores['test_score'].mean())

Train : 0.886136193834053
Valid : 0.8714622047827053


----
#### Histogram Gradient Boosting
- 훈련데이터를 256개의 구간으로 나누어서 훈련시키는 방법
- 특성의 범위가 제한되어 있어 빠른 속도를 제공한다.
- 제한된 구간이므로 과대적합을 방지한다.

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier()

scores = cross_validate(
        hgb,
        train_input,
        train_target,
        return_train_score=True,
        n_jobs= -1
)

print('Train :', scores['train_score'].mean())
print('Valid :', scores['test_score'].mean())


Train : 0.930536541746549
Valid : 0.8780021470348707


----
LightGBM
- Gradient Boosting에서 출발

In [ ]:
# !pip install lightgbm

In [28]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier()

scores = cross_validate(
        lgb,
        train_input,
        train_target,
        return_train_score=True,
        n_jobs= -1
)

print('Train :', scores['train_score'].mean())
print('Valid :', scores['test_score'].mean())

Train : 0.934577605325741
Valid : 0.8805047382838529
